# Week 1 · Exercise 1: pandas, NumPy, pipelines and cross-validation

**Goal:** predict which loan applicants default, and do it without fooling yourself.

Data: `make_credit_applicants()` from the `classical_ml` package, a synthetic set of 5,000 loan
applicants (~12% default). It is generated from a known formula, so we know what the "right"
answer looks like. Next week you swap in a real dataset.

The finished, tested code already lives in `classical_ml/src/classical_ml/`. This notebook is the
scratchpad where you see *why* each piece is there. Look for **✏️ Try it** cells.

## 1. pandas tour: look before you model

In [ ]:
import numpy as np
import pandas as pd

from classical_ml.datasets import CATEGORICAL_FEATURES, NUMERIC_FEATURES, TARGET, make_credit_applicants

df = make_credit_applicants()
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(1)

Missing values per column. Any model you fit has to cope with these.

In [ ]:
df.isna().mean().round(3)

`groupby` is the workhorse. Default rate by employment type: does it match intuition?
(`dropna=False` keeps the rows where employment type is missing as their own group.)

In [ ]:
df.groupby("employment_type", dropna=False)[TARGET].agg(["mean", "count"]).sort_values("mean")

**✏️ Try it:** compute the default rate by `region` and by `owns_home`. One of these should
look like noise (the generator ignores it). Which one?

In [ ]:
# your code here

## 2. NumPy: vectorise, don't loop

Debt-to-income (loan ÷ income) is a classic credit feature. Compare a Python loop with one
vectorised expression.

In [ ]:
loan = df["loan_amount"].to_numpy()
income = df["annual_income"].to_numpy()

def dti_loop(loan, income):
    out = []
    for l, i in zip(loan, income):
        out.append(l / i)
    return np.array(out)

%timeit dti_loop(loan, income)
%timeit loan / income
np.allclose(dti_loop(loan, income), loan / income, equal_nan=True)

Same answer, roughly 100× faster. NumPy runs the loop in C over a contiguous array. Note that
NaN income gives NaN debt-to-income: missing values propagate.

**✏️ Try it:** add a `debt_to_income` column to `df` and check its default rate by quartile
with `pd.qcut(df["debt_to_income"], 4)`.

In [ ]:
# your code here

## 3. Supervised vs unsupervised

- **Supervised:** you have labels (`defaulted`) and learn a mapping features → label.
- **Unsupervised:** no labels. You look for structure, e.g. groups of similar applicants.

Same data, two questions. First, supervised: logistic regression predicting default.

In [ ]:
from sklearn.linear_model import LogisticRegression

from classical_ml.pipeline import build_preprocessor

X = df.drop(columns=TARGET)
y = df[TARGET]

X_matrix = build_preprocessor().fit_transform(X)   # numbers only, no NaNs (see section 5)
clf = LogisticRegression(max_iter=1000).fit(X_matrix, y)
clf.predict_proba(X_matrix[:5])[:, 1].round(3)    # P(default) for the first 5 applicants

Unsupervised: k-means puts applicants into 4 clusters *without ever seeing `defaulted`*.

In [ ]:
from sklearn.cluster import KMeans

clusters = KMeans(n_clusters=4, n_init="auto", random_state=0).fit_predict(X_matrix)
df.assign(cluster=clusters).groupby("cluster")[TARGET].agg(["mean", "count"])

If cluster default rates differ, the clusters found real structure that happens to relate to
risk. That is a common use of unsupervised learning: segmentation, then checking what segments mean.

(Yes, section 3 fit the preprocessor on *all* rows. Fine for a demo, a bug for evaluation. That's next.)

## 4. Train / validation / test, and leakage

- **Train:** fit the model.
- **Validation:** compare models and tune settings. You'll look at it many times.
- **Test:** touch it **once**, at the end, to report a number you didn't optimise against.

The repo already has a stratified splitter (keeps the 12% default rate in every split):

In [ ]:
from classical_ml.data import train_val_test_split

train, val, test = train_val_test_split(df, target=TARGET)
pd.DataFrame({s: [len(d), d[TARGET].mean()] for s, d in [("train", train), ("val", val), ("test", test)]},
             index=["rows", "default rate"]).round(3)

**Leakage** = information from the evaluation data sneaking into training, so your score is
better than reality. The nastiest kind hides in preprocessing.

Demo: 200 rows of **pure random noise**, 5,000 features, random labels. No model can beat 50%.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
X_noise = rng.normal(size=(200, 5000))
y_noise = rng.integers(0, 2, size=200)

# WRONG: pick the 20 "best" features using ALL rows, then cross-validate
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(LogisticRegression(), X_selected, y_noise, cv=5).mean()

# RIGHT: feature selection inside the pipeline, so it's refit on each training fold only
honest = cross_val_score(
    make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression()), X_noise, y_noise, cv=5
).mean()

print(f"leaky accuracy:  {leaky:.2f}")
print(f"honest accuracy: {honest:.2f}")

The leaky version "finds signal" in noise because the feature selector already saw the
validation folds' labels. **Rule: anything that learns from data goes inside the pipeline.**
That includes imputers (they learn medians) and scalers (they learn means).

## 5. `Pipeline` + `ColumnTransformer`

`build_preprocessor()` (in `classical_ml/pipeline.py`) routes columns by type:

| columns | steps |
|---|---|
| numeric | median imputation → standard scaling |
| categorical | most-frequent imputation → one-hot encoding |

`build_pipeline()` puts a model after it. Fitting the pipeline fits every step on the training
data only; predicting just applies them.

In [ ]:
from classical_ml.pipeline import build_pipeline

pipe = build_pipeline()
pipe

In [ ]:
pipe.fit(train.drop(columns=TARGET), train[TARGET])
pipe.named_steps["preprocess"].get_feature_names_out()

## 6. k-fold cross-validation

One validation split gives one noisy number. 5-fold CV trains 5 times, each time holding out a
different fifth, and gives you a mean ± spread. Stratified so each fold keeps the 12% default rate.

Metric: **ROC-AUC**, the probability the model ranks a random defaulter above a random
non-defaulter (0.5 = coin flip, 1.0 = perfect). Accuracy is useless here: predicting "nobody
defaults" scores 88%. Week 2 goes deep on metrics.

In [ ]:
from sklearn.dummy import DummyClassifier

from classical_ml.pipeline import cross_validate

train_val = pd.concat([train, val])
X_tv, y_tv = train_val.drop(columns=TARGET), train_val[TARGET]

print("dummy:   ", cross_validate(build_pipeline(DummyClassifier()), X_tv, y_tv))
print("logistic:", cross_validate(build_pipeline(), X_tv, y_tv))

Always compare with a dummy baseline. "0.75 AUC" means nothing until you know chance is 0.5.

Finally, **once**, the test set:

In [ ]:
from sklearn.metrics import roc_auc_score

final = build_pipeline().fit(X_tv, y_tv)
test_auc = roc_auc_score(test[TARGET], final.predict_proba(test.drop(columns=TARGET))[:, 1])
print(f"test ROC-AUC: {test_auc:.3f}")

## ✏️ Your turn

Do these here first, then move anything reusable into the package **with a test** (that's the
Week 1 deliverable).

1. **Model swap.** Pass `RandomForestClassifier(random_state=0)` to `build_pipeline()` and CV it.
   Does it beat logistic regression? Is the gap bigger than the ± spread?
2. **Feature engineering.** The generator uses debt-to-income, but the model only sees loan and
   income separately. Add a `debt_to_income` feature (hint: a `FunctionTransformer` in the
   numeric branch, or a new column in the dataset). How much does CV AUC move?
3. **Drop the noise.** Remove `region` from the categorical features. Does the score change?
4. **Break it on purpose.** Fit `StandardScaler` on the full `X_tv` *before* CV and compare.
   Why is the effect tiny here but huge in the noise demo? (Hint: what does each step learn?)
5. **Look inside.** After fitting `final`, read `final.named_steps["model"].coef_` next to the
   feature names. Do the signs match the generator in `datasets.py`?
6. **Write-up.** Start `notes/week-01.md`: what you built, one number that moved, what confused
   you, one interview question you can now answer (try: *"What is data leakage and how do you
   prevent it?"*).